In [1]:
import h5py
import torch
import numpy as np
import os
import json

import keyrank_rs

import torch.nn as nn

from tqdm import tqdm

In [2]:
if os.name == "nt":
    DATAFOLDER = "C:/Data"
else:
    DATAFOLDER = "/mnt/c/Data"

val_test_hdf = h5py.File(f"{DATAFOLDER}/simpleserial-aes-fix-500-diff.hdf5")

val_test_traces = torch.Tensor(np.array(val_test_hdf['trace']))
val_test_plaintexts = torch.Tensor(np.array(val_test_hdf['data']))
val_test_keys = torch.Tensor(np.array(val_test_hdf['key']))

device = torch.device("cuda")

In [3]:
print(val_test_traces.shape)
print(val_test_plaintexts.shape)
print(val_test_keys.shape)

torch.Size([1000, 500, 5000])
torch.Size([1000, 500, 32])
torch.Size([1000, 16])


In [4]:
def metadata_best_epoch(model_name) -> int:
    with open(f"models/{model_name}/metadata.json") as f:
        metadata = json.load(f)
        val_scores = metadata["scores"][1]
        best_epoch = np.array(val_scores).argmin()
    return best_epoch.item()

def get_traces_mean_std(trace_start, trace_end):
    """Load the mean and std of the training trace set within the given interval"""
    with open(f"misc/standardization/trace{trace_start}_{trace_end}.json") as f:
        info = json.load(f)
        mean = info['training_traces_mean']
        std = info['training_traces_std']

    return mean, std

In [5]:
IMPL = "fixslice"
ARCH = "zhang"
PREDICTION_TARGET = "sbox"
TARGET_BYTE_IDX = 2
TRACE_START = 400
TRACE_END = 1500
SEED = 777

model_name = f"{IMPL}-{PREDICTION_TARGET}-byte{TARGET_BYTE_IDX}-{ARCH}-{TRACE_START}_{TRACE_END}-s{SEED}"

epoch = metadata_best_epoch(model_name)

model_path = f"models/{model_name}/epoch{epoch}.pt"
print(model_path)

model = torch.load(model_path).to()

models/fixslice-sbox-byte2-zhang-400_1500-s777/epoch16.pt


C:\Users\Ulrik\AppData\Local\Temp\ipykernel_12720\2434003031.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path).to()


In [6]:
sample = 259

traces_mean, traces_std = get_traces_mean_std(TRACE_START, TRACE_END)

traces = (val_test_traces[sample, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
plaintexts_B1 = val_test_plaintexts[sample, :, :16] # first plaintext block
plaintexts_B2 = val_test_plaintexts[sample, :, 16:] # second plaintext block
key = val_test_keys[sample]

print(traces.shape)
print(plaintexts_B1.shape)
print(key.shape)

torch.Size([500, 1100])
torch.Size([500, 16])
torch.Size([16])


In [ ]:
"""Compute traces needed for 99% accurate full key recovery using single model"""

log_softmax = nn.LogSoftmax(dim=1)

full_recovery_traces = []

for training_byte in range(16):

    model_name = f"{IMPL}-{PREDICTION_TARGET}-byte{training_byte}-{ARCH}-{TRACE_START}_{TRACE_END}-s{SEED}"

    epoch = metadata_best_epoch(model_name)

    model_path = f"models/{model_name}/epoch{epoch}.pt"
    print(model_path)

    model = torch.load(model_path).to()

    # Sample_idx, n_traces
    success_matrix = torch.zeros(500,500)


    for sample_idx in tqdm(range(0,500)):

        traces_ = (val_test_traces[sample_idx, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
        plaintexts_B1_ = val_test_plaintexts[sample_idx, :, :16] # first plaintext block
        plaintexts_B2_ = val_test_plaintexts[sample_idx, :, 16:] # second plaintext block
        true_key_ = val_test_keys[sample_idx].long()

        sbox_scores = model(traces_.to(device))
        numpy_sbox_scores = sbox_scores.detach().cpu().numpy()

        full_guesses = torch.zeros(500,16)

        for subkey in range(16):
            plaintext_B1_bytes = plaintexts_B1_[:, subkey]
            plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

            plaintext_B2_bytes = plaintexts_B2_[:, subkey]
            plaintext_B2_bytes = plaintext_B2_bytes.long().detach().cpu().numpy().squeeze()

            numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_sbox_scores)
            #numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_sbox_scores)

            x = torch.Tensor(numpy_keyscores1)# + numpy_keyscores2)
            x = log_softmax(x)
            x = x.cumsum(dim=0)
            guesses = x.argmax(dim=1)
            full_guesses[:, subkey] = guesses

        
        successes = (full_guesses.long() == true_key_).all(dim=1)
        success_matrix[sample_idx, :] = successes


    full_recovery_traces.append((success_matrix.sum(dim=0) >= 495).nonzero()[0].item() + 1)

full_recovery_traces

C:\Users\Ulrik\AppData\Local\Temp\ipykernel_15656\2848987065.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path).to()
  0%|          | 0/500 

models/fixslice-sbox-byte0-zhang-400_1500-s777/epoch14.pt


  0%|          | 0/500 [00:00<?, ?it/s]

models/fixslice-sbox-byte1-zhang-400_1500-s777/epoch32.pt


  0%|          | 0/500 [00:00<?, ?it/s]

...

models/fixslice-sbox-byte14-zhang-400_1500-s777/epoch13.pt


  0%|          | 0/500 [00:00<?, ?it/s]

models/fixslice-sbox-byte15-zhang-400_1500-s777/epoch14.pt


100%|██████████| 500/500 [00:30<00:00, 16.60it/s]


[119,
 186,
 122,
 126,
 118,
 115,
 109,
 118,
 121,
 127,
 129,
 122,
 206,
 151,
 156,
 131]

In [ ]:
"""Compute traces needed for 99% accuracy on individual subkeys using single model"""


log_softmax = nn.LogSoftmax(dim=1)

model_key_traces_needed = []

for training_byte in range(16):

    model_name = f"{IMPL}-{PREDICTION_TARGET}-byte{training_byte}-{ARCH}-{TRACE_START}_{TRACE_END}-s{SEED}"

    epoch = metadata_best_epoch(model_name)

    model_path = f"models/{model_name}/epoch{epoch}.pt"
    print(model_path)

    model = torch.load(model_path).to()


    # subkey, sample_idx, n_traces
    success_matrix = torch.zeros(16,500,500)

    for sample_idx in tqdm(range(0,500)):

        traces_ = (val_test_traces[sample_idx, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
        plaintexts_B1_ = val_test_plaintexts[sample_idx, :, :16] # first plaintext block
        plaintexts_B2_ = val_test_plaintexts[sample_idx, :, 16:] # second plaintext block
        true_key_ = val_test_keys[sample_idx].long()

        sbox_scores = model(traces_.to(device))
        numpy_sbox_scores = sbox_scores.detach().cpu().numpy()

        for subkey in range(16):
            plaintext_B1_bytes = plaintexts_B1_[:, subkey]
            plaintext_B1_bytes = plaintext_B1_bytes.long().detach().cpu().numpy().squeeze()

            plaintext_B2_bytes = plaintexts_B2_[:, subkey]
            plaintext_B2_bytes = plaintext_B2_bytes.long().detach().cpu().numpy().squeeze()

            numpy_keyscores1 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B1_bytes, numpy_sbox_scores)
            numpy_keyscores2 = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_B2_bytes, numpy_sbox_scores)

            x = torch.Tensor(numpy_keyscores1 + numpy_keyscores2)
            x = log_softmax(x)
            x = x.cumsum(dim=0)
            guesses = x.argmax(dim=1)

            success = (guesses.long() == true_key_[subkey])
            success_matrix[subkey, sample_idx] = success


    n_traces_needed = []

    for subkey in range(16):
        n_traces_needed.append((success_matrix[subkey].sum(dim=0) >= 495).nonzero()[0].item() + 1)

    model_key_traces_needed.append(n_traces_needed)

models/fixslice-sbox-byte0-zhang-400_1500-s777/epoch14.pt


C:\Users\Ulrik\AppData\Local\Temp\ipykernel_12720\983831177.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path).to()
100%|██████████| 500/500

models/fixslice-sbox-byte1-zhang-400_1500-s777/epoch32.pt


  0%|          | 0/500 [00:00<?, ?it/s]

models/fixslice-sbox-byte2-zhang-400_1500-s777/epoch16.pt


...

100%|██████████| 500/500 [00:35<00:00, 14.11it/s]


models/fixslice-sbox-byte14-zhang-400_1500-s777/epoch13.pt


100%|██████████| 500/500 [00:35<00:00, 14.19it/s]


models/fixslice-sbox-byte15-zhang-400_1500-s777/epoch14.pt


100%|██████████| 500/500 [00:36<00:00, 13.88it/s]


In [8]:
for fdgs in model_key_traces_needed:
    print(fdgs)

[30, 18, 27, 22, 29, 25, 18, 25, 27, 29, 21, 16, 51, 37, 36, 33]
[14, 8, 14, 14, 34, 31, 18, 27, 29, 23, 25, 22, 64, 43, 60, 41]
[32, 22, 23, 25, 31, 24, 18, 23, 24, 29, 19, 15, 51, 35, 40, 32]
[34, 22, 25, 21, 29, 25, 22, 24, 24, 29, 20, 18, 44, 33, 36, 32]
[32, 20, 29, 20, 26, 22, 18, 22, 25, 26, 18, 16, 49, 39, 40, 37]
[34, 21, 28, 21, 31, 22, 20, 21, 24, 27, 18, 16, 47, 35, 36, 32]
[27, 14, 23, 19, 23, 21, 15, 20, 24, 25, 16, 13, 44, 33, 31, 30]
[36, 20, 25, 24, 29, 26, 22, 24, 25, 29, 21, 20, 49, 33, 40, 32]
[32, 19, 22, 19, 30, 22, 18, 21, 22, 28, 17, 13, 51, 34, 38, 32]
[29, 16, 26, 19, 25, 23, 18, 21, 24, 20, 17, 14, 53, 37, 40, 36]
[24, 16, 20, 18, 29, 20, 16, 19, 19, 23, 14, 12, 49, 37, 38, 27]
[24, 18, 23, 19, 21, 18, 15, 18, 20, 19, 13, 10, 47, 32, 38, 27]
[70, 50, 58, 49, 61, 71, 48, 63, 64, 81, 53, 48, 72, 62, 63, 64]
[47, 33, 37, 36, 45, 48, 34, 38, 43, 50, 33, 30, 62, 43, 53, 38]
[53, 33, 34, 30, 42, 43, 31, 35, 45, 46, 30, 29, 57, 48, 46, 43]
[44, 32, 35, 29, 37, 45, 2

In [17]:
with open("misc/subkey_comparison/plaintext1_full_grid.json", 'w') as f:
    json.dump({
        "grid" : model_key_traces_needed,
    },
    f,
    indent=4
    )

In [9]:
print("Training subkey / Target subkey: &",  " & ".join([f"${n}$" for n in range(16)]), r"\\")

for i,n_traces_needed in enumerate(model_key_traces_needed):
    print("\\hline")
    print(f"${i}$ &", " & ".join([f"{n_traces:.0f}" for n_traces in n_traces_needed]), r"\\")


#for idx, (mean, n99acc) in enumerate(zip(mean_traces_needed, traces_needed_99acc)):
#    print(f"Subkey {idx:02}, mean: {mean:.03f}, traces needed for 99%: {n99acc}")

Training subkey / Target subkey: & $0$ & $1$ & $2$ & $3$ & $4$ & $5$ & $6$ & $7$ & $8$ & $9$ & $10$ & $11$ & $12$ & $13$ & $14$ & $15$ \\
\hline
$0$ & 30 & 18 & 27 & 22 & 29 & 25 & 18 & 25 & 27 & 29 & 21 & 16 & 51 & 37 & 36 & 33 \\
\hline
$1$ & 14 & 8 & 14 & 14 & 34 & 31 & 18 & 27 & 29 & 23 & 25 & 22 & 64 & 43 & 60 & 41 \\
\hline
$2$ & 32 & 22 & 23 & 25 & 31 & 24 & 18 & 23 & 24 & 29 & 19 & 15 & 51 & 35 & 40 & 32 \\
\hline
$3$ & 34 & 22 & 25 & 21 & 29 & 25 & 22 & 24 & 24 & 29 & 20 & 18 & 44 & 33 & 36 & 32 \\
\hline
$4$ & 32 & 20 & 29 & 20 & 26 & 22 & 18 & 22 & 25 & 26 & 18 & 16 & 49 & 39 & 40 & 37 \\
\hline
$5$ & 34 & 21 & 28 & 21 & 31 & 22 & 20 & 21 & 24 & 27 & 18 & 16 & 47 & 35 & 36 & 32 \\
\hline
$6$ & 27 & 14 & 23 & 19 & 23 & 21 & 15 & 20 & 24 & 25 & 16 & 13 & 44 & 33 & 31 & 30 \\
\hline
$7$ & 36 & 20 & 25 & 24 & 29 & 26 & 22 & 24 & 25 & 29 & 21 & 20 & 49 & 33 & 40 & 32 \\
\hline
$8$ & 32 & 19 & 22 & 19 & 30 & 22 & 18 & 21 & 22 & 28 & 17 & 13 & 51 & 34 & 38 & 32 \\
\hline
$9$ & 29 &

In [ ]:
"""Export data from one attack on a single full key"""


n_traces = 42

true_key =  key.long().tolist()
guesses = []

attack_data_folder = f"misc/attack_data/key{sample}"
os.makedirs(os.path.dirname(attack_data_folder), exist_ok=True)

np.save(f"{attack_data_folder}/sbox_scores.npy", numpy_sbox_scores_slice)
np.save(f"{attack_data_folder}/plaintexts.npy", plaintexts_B1[:n_traces])

for subkey in range(16):

    plaintext_bytes = plaintexts_B1[:n_traces, subkey]
    plaintext_bytes = plaintext_bytes.long().detach().cpu().numpy().squeeze()

    numpy_keyscores = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_bytes, numpy_sbox_scores_slice)

    np.save(f"{attack_data_folder}/keybyte{subkey}_scores.npy", numpy_keyscores)

    x = torch.Tensor(numpy_keyscores).softmax(dim=1)
    x = x.log()
    x = x.sum(dim=0)
    x = x.argmax(dim=0)

    guesses.append(x.item())



print("True key:", true_key)
print("Full attack:",guesses)


attack_info = {
    "implementation" : IMPL,
    "architecture" : ARCH,
    "target_variable" : PREDICTION_TARGET,
    "training_target_byte" : TARGET_BYTE_IDX,
    "trace_interval_start" : TRACE_START,
    "trace_interval_end" : TRACE_END,
    "testing_set_index" : sample,
    "attack_traces" : n_traces,
    "true_key" : true_key,
    "attack_output" : guesses,
}

with open(f"{attack_data_folder}/attack_info.json", 'w') as f:
    json.dump(attack_info, f, indent=4)

True key: [66, 59, 237, 182, 158, 214, 87, 231, 242, 124, 155, 22, 62, 118, 10, 48]
Full attack: [66, 59, 237, 182, 158, 214, 87, 231, 242, 124, 155, 22, 62, 118, 10, 48]
